## import

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import numpy.matlib    
import pandas as pd
import scipy as sc
import csv
import math
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import isspmatrix, csc_matrix, csr_matrix
from scipy.sparse.linalg import eigsh, svds
from scipy.linalg import svd
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder
import io
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch import tensor
from torch.nn import functional as F 

##  SOURCE DATASET LOADING

In [2]:
#rating matrix
rating_matrix_video = np.loadtxt("rating_matrix_video_2k.csv", delimiter=",")

#users
temp_d = pd.read_csv('new_user_ids_video_2k.csv')
temp_np = temp_d.to_numpy()
temp_np = np.delete(temp_np, 0, 1)
new_user_ids_video_2k = temp_np.reshape(temp_np.shape[0],)

print("\nSource user ids:\n")
print(new_user_ids_video_2k)
print("\nTotal Number of source users:")
print(len(new_user_ids_video_2k))

#items
temp_d = pd.read_csv('item_ids_video_2k_rated_unrated.csv')
temp_np = temp_d.to_numpy()
temp_np = np.delete(temp_np, 0, 1)
item_ids_video_2k_rated_unrated = temp_np.reshape(temp_np.shape[0],)

print("\n\nsource item ids:\n")
print(item_ids_video_2k_rated_unrated)
print("\n\nTotal Number of source items:\n")
print(len(item_ids_video_2k_rated_unrated))




Source user ids:

['A234NQ63JSJDQA' 'A30W79WBVU4AAK' 'A2DVN9KQTWN8U9' ... 'A2WS0REGOWR9HY'
 'A3T5YE6V0Q599I' 'AY0B3WP6G0OO4']

Total Number of source users:
9000


source item ids:

['B004XYN4NY' 'B000B9RI00' 'B00008URUB' ... 'B000AYDRC4' 'B00576U8CC'
 'B00005N9A7']


Total Number of source items:

7500


## TARGET DATASET LOADING

In [3]:
#rating matrix
rating_matrix_mt = np.loadtxt("rating_matrix_mt_2k.csv", delimiter=",")

#users
temp_d = pd.read_csv('new_user_ids_mt_2k.csv')
temp_np = temp_d.to_numpy()
temp_np = np.delete(temp_np, 0, 1)
new_user_ids_mt_2k = temp_np.reshape(temp_np.shape[0],)

print("target user ids:\n\n")
print(new_user_ids_mt_2k)
print("\n\nTotal Number of target users:")
print(len(new_user_ids_mt_2k))

#items
temp_d = pd.read_csv('item_ids_mt_2k_rated_unrated.csv')
temp_np = temp_d.to_numpy()
temp_np = np.delete(temp_np, 0, 1)
item_ids_mt_2k_rated_unrated = temp_np.reshape(temp_np.shape[0],)

print("\n\nTarget item ids:\n")
print(item_ids_mt_2k_rated_unrated)
print("\n\nTotal Number of target item:\n")
print(len(item_ids_mt_2k_rated_unrated))

target user ids:


['A10Z3FTSP8X91M' 'A10U5D2T50JREX' 'A1UCR4XL6WPDO3' ... 'A274R1IXVMVKAG'
 'A101R7OO4IRDTH' 'AIU4DRBTNGCQ8']


Total Number of target users:
7000


Target item ids:

['B0018PH3J2' '6302787068' 'B000VE4UQA' ... '630218164X' 'B005M0X078'
 'B000VBJDGK']


Total Number of target item:

11480


## encoding

In [4]:
all_user_ids = np.concatenate((new_user_ids_video_2k, new_user_ids_mt_2k))

unique_user_ids = np.unique(all_user_ids)
total_num_users = len(unique_user_ids)
user_id_map = {str(user_id): idx + 1 for idx, user_id in enumerate(unique_user_ids)}

def map_user_ids(user_ids):
  vectorized_map = np.vectorize(lambda x: user_id_map.get(x))
  return vectorized_map(user_ids)

new_user_ids_video_2k = map_user_ids(new_user_ids_video_2k)
new_user_ids_mt_2k = map_user_ids(new_user_ids_mt_2k)

In [5]:
print(new_user_ids_video_2k)
print(new_user_ids_mt_2k)

[ 5430  8378  6343 ...  8005 10857 13828]
[ 1349  1167  4672 ...  5753   191 12480]


In [6]:
all_item_ids = np.concatenate((item_ids_video_2k_rated_unrated, item_ids_mt_2k_rated_unrated))

unique_item_ids = np.unique(all_item_ids)

total_num_items = len(unique_item_ids)

user_id_map = {str(user_id): idx + 1 for idx, user_id in enumerate(unique_item_ids)}

def map_user_ids(user_ids):
  vectorized_map = np.vectorize(lambda x: user_id_map.get(x))
  return vectorized_map(user_ids)

item_ids_video_2k_rated_unrated = map_user_ids(item_ids_video_2k_rated_unrated)
item_ids_mt_2k_rated_unrated = map_user_ids(item_ids_mt_2k_rated_unrated)

In [7]:
print(item_ids_video_2k_rated_unrated)
print(item_ids_mt_2k_rated_unrated)

[15466  7828  5379 ...  7753 15783  4266]
[11127  1630 10226 ...  1447 16127 10216]


## OVERLAP USERS

In [8]:
overlapping_user_ids = np.intersect1d(new_user_ids_video_2k, new_user_ids_mt_2k)
print(overlapping_user_ids)
print(overlapping_user_ids.shape)

[   1    2    3 ... 2134 2135 2136]
(2000,)


In [9]:
print(overlapping_user_ids.dtype)

int32


## NEGATIVE SAMPLES

In [10]:
#SOURCE

source_user_negative_items = {}
for user_index, user_id in enumerate(new_user_ids_video_2k):
    source_user_negative_items[user_id] = []
    for item_index, item_id in enumerate(item_ids_video_2k_rated_unrated):
        if rating_matrix_video[user_index, item_index] == 0:
            source_user_negative_items[user_id].append(item_id)
            

used_negative_samples = {user_id: set() for user_id in new_user_ids_video_2k}

source_negative_samples = []
for user_index, user_id in enumerate(new_user_ids_video_2k):
    for item_index, item_id in enumerate(item_ids_video_2k_rated_unrated):
        if rating_matrix_video[user_index, item_index] == 1:
            negative_candidates = list(set(source_user_negative_items[user_id]) - used_negative_samples[user_id])
            if negative_candidates:
                negative_sample = random.choice(negative_candidates)
                used_negative_samples[user_id].add(negative_sample)
                rating = rating_matrix_video[user_index, item_index]
                source_negative_samples.append([user_id, item_id, negative_sample, rating])

           
source_negative_samples_2k_df = pd.DataFrame(source_negative_samples, columns=["user_id", "item_id", "negative_item_id", "rating"])

source_negative_samples_2k_df.to_csv('source_negative_samples_2k.csv', index=False)

print(source_negative_samples_2k_df.head())

print(source_negative_samples_2k_df.dtypes)

   user_id  item_id  negative_item_id  rating
0     1395     6809             16564     1.0
1     7741     4781             10383     1.0
2     7570    13165              6391     1.0
3     1753     6366             11637     1.0
4     9318    15329             13260     1.0
user_id               int32
item_id               int32
negative_item_id      int32
rating              float64
dtype: object


In [11]:
print(source_negative_samples_2k_df)

      user_id  item_id  negative_item_id  rating
0        1395     6809             16564     1.0
1        7741     4781             10383     1.0
2        7570    13165              6391     1.0
3        1753     6366             11637     1.0
4        9318    15329             13260     1.0
...       ...      ...               ...     ...
1677    11240    11353              3029     1.0
1678     2422    16564             12691     1.0
1679     9882     8971             15999     1.0
1680     9916    18180              8555     1.0
1681     4189     9850              5572     1.0

[1682 rows x 4 columns]


In [12]:
#TARGET

target_user_negative_items = {}
for user_index, user_id in enumerate(new_user_ids_mt_2k):
    target_user_negative_items[user_id] = []
    for item_index, item_id in enumerate(item_ids_mt_2k_rated_unrated):
        if rating_matrix_mt[user_index, item_index] == 0:
            target_user_negative_items[user_id].append(item_id)

used_negative_samples = {user_id: set() for user_id in new_user_ids_mt_2k}

target_negative_samples = []
for user_index, user_id in enumerate(new_user_ids_mt_2k):
    for item_index, item_id in enumerate(item_ids_mt_2k_rated_unrated):
        if rating_matrix_mt[user_index, item_index] == 1:
            negative_candidates = []
            if user_id in source_user_negative_items:
                negative_candidates = list(set(source_user_negative_items[user_id]) - used_negative_samples[user_id])
            
            if negative_candidates:
                negative_sample = random.choice(negative_candidates)
                used_negative_samples[user_id].add(negative_sample)
                rating = rating_matrix_mt[user_index, item_index]
                target_negative_samples.append([user_id, item_id, negative_sample, rating])



target_negative_samples_2k_df = pd.DataFrame(target_negative_samples, columns=["user_id", "item_id", "negative_item_id","rating"])

target_negative_samples_2k_df.to_csv('target_negative_samples_2k.csv', index=False)

print(target_negative_samples_2k_df.head())

   user_id  item_id  negative_item_id  rating
0     1167     9305              2796     1.0
1     1167     2483             15470     1.0
2     1167     3201             14809     1.0
3     2004    12357             12756     1.0
4      162    17931             13581     1.0


In [13]:
print(target_negative_samples_2k_df)

     user_id  item_id  negative_item_id  rating
0       1167     9305              2796     1.0
1       1167     2483             15470     1.0
2       1167     3201             14809     1.0
3       2004    12357             12756     1.0
4        162    17931             13581     1.0
..       ...      ...               ...     ...
579     1843     8718             15606     1.0
580      232    18067             14215     1.0
581     2048    18081              9752     1.0
582     2048    17459             14445     1.0
583     2048    17827             11377     1.0

[584 rows x 4 columns]


In [14]:
# overlapping_user= np.intersect1d(source_negative_samples_df['user_id'], target_negative_samples_df['user_id'])
# print(overlapping_user)
# print(overlapping_user.shape)

## Dataloader class

In [15]:
class GetDataset(torch.utils.data.Dataset):
    
    def __init__(self, data_df):
        self.data = data_df 

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        user_id = torch.tensor(self.data.iloc[idx]["user_id"], dtype=torch.int64)
        item_id = torch.tensor(self.data.iloc[idx]["item_id"], dtype=torch.int64)
        negative_item_id = torch.tensor(self.data.iloc[idx]["negative_item_id"], dtype=torch.int64)
        rating = torch.tensor(self.data.iloc[idx]["rating"], dtype=torch.float)
        return user_id, item_id, negative_item_id, rating

## mapping layer

In [16]:
class MappingLayer(nn.Module):
    def __init__(self, embedding_size, mlp_hidden_size, activation='Tanh', dropout=0):
        super(MappingLayer, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(embedding_size, mlp_hidden_size[0]), 
            nn.Tanh(),
            nn.Linear(mlp_hidden_size[0], embedding_size) 
        )

    def forward(self, x):
        return self.layers(x)

In [17]:
torch.manual_seed(42)

## train test split

In [18]:
def train_test_split_df(data_df,test_size=0.3):
    train, test = train_test_split(data_df, test_size=test_size, shuffle=True)

    return train,test

In [19]:
source_train_df,source_test_df = train_test_split_df(source_negative_samples_2k_df, test_size=0.3)

source_train_loader = DataLoader(GetDataset(source_train_df), batch_size=64, shuffle=True)
source_test_loader = DataLoader(GetDataset(source_test_df), batch_size=64, shuffle=False)

In [20]:
target_train_df,target_test_df = train_test_split_df(target_negative_samples_2k_df, test_size=0.3)

target_train_loader = DataLoader(GetDataset(target_train_df), batch_size=64, shuffle=True)
target_test_loader = DataLoader(GetDataset(target_test_df), batch_size=64, shuffle=False)

In [21]:
overlap_tensor = torch.tensor(overlapping_user_ids)

overlap_user_df = pd.DataFrame({"user_id": overlapping_user_ids})
print(overlap_user_df)


overlap_loader = DataLoader(overlap_user_df,batch_size=64,shuffle=True)

      user_id
0           1
1           2
2           3
3           4
4           5
...       ...
1995     2132
1996     2133
1997     2134
1998     2135
1999     2136

[2000 rows x 1 columns]


## dataset class

In [22]:
class SSCDRDataset(torch.utils.data.Dataset):
    def __init__(self, source_data, target_data, overlap_data,total_num_users,total_num_items):
        self.source_data = source_data
        self.target_data = target_data
        self.overlap_data = overlap_data

       
        self.source_num_users = len(source_data['user_id'].unique())
        self.source_num_items = len(source_data['item_id'].unique())

        
        self.target_num_users = len(target_data['user_id'].unique())
        self.target_num_items = len(target_data['item_id'].unique())

        
        required_source_cols = ['user_id', 'item_id', 'negative_item_id']
        required_target_cols = ['user_id', 'item_id', 'negative_item_id']
        assert all(col in source_data.columns for col in required_source_cols), f"Source data must contain columns: {', '.join(required_source_cols)}"
        assert all(col in target_data.columns for col in required_target_cols), f"Target data must contain columns: {', '.join(required_target_cols)}"

        
        self.source_user_ids = source_data['user_id']
        self.source_item_ids = source_data['item_id']
        self.source_negative_item_ids = source_data['negative_item_id']
        self.target_user_ids = target_data['user_id']
        self.target_item_ids = target_data['item_id']
        self.target_negative_item_ids = target_data['negative_item_id']

      
 
        
        # self.overlapped_num_items = 0
        self.num_overlap_users = len(overlap_data)
        self.num_overlap_items = 0
        self.overlap_user_ids = overlap_data
       

        # Total number of users and items (considering both domains and overlap)
        self.total_num_users = total_num_users
        self.total_num_items = total_num_items

In [23]:
dataset = SSCDRDataset(source_train_df,target_train_df,overlap_user_df,total_num_users,total_num_items)

## model class

In [24]:
class SSCDR(nn.Module):

    # input_type = InputType.PAIRWISE
    # type = ModelType.CROSSDOMAIN

    def __init__(self,config,dataset):
        super(SSCDR, self).__init__()

        self.dataset = dataset
        # <----------load parameters info-------->
        self.device = config['device']
        self.embedding_size = config['embedding_size']
        self.lamda = config['lambda']
        self.margin = config['margin']
        self.mlp_hidden_size = config['mlp_hidden_size']
        self.mode = 'overlap_users'
        self.mapping_layer =MappingLayer(embedding_size=self.embedding_size, mlp_hidden_size=self.mlp_hidden_size, activation='Tanh', dropout=0)

      
        # <------------- source details -------------->
        self.SOURCE_USER_ID = dataset.source_user_ids
        self.SOURCE_ITEM_ID = dataset.source_item_ids
        self.SOURCE_NEG_ITEM_ID = dataset.source_negative_item_ids
        self.source_num_users = dataset.source_num_users
        self.source_num_items = dataset.source_num_items



        # <------------- Target details -------------->
        self.TARGET_USER_ID = dataset.target_user_ids
        self.TARGET_ITEM_ID = dataset.target_item_ids
        self.TARGET_NEG_ITEM_ID = dataset.target_negative_item_ids
        self.target_num_users = dataset.target_num_users
        self.target_num_items = dataset.target_num_items

        # load both dataset info
        self.total_num_users = dataset.total_num_users
        self.total_num_items = dataset.total_num_items

        self.overlapped_num_users = dataset.num_overlap_users
        self.overlapped_num_items = dataset.num_overlap_items

        self.overlap_user_ids = dataset.overlap_user_ids
        
    
        # <------------------- define layers and loss ---------------------->

        self.source_user_embedding = torch.nn.Embedding(self.total_num_users, self.embedding_size)
        self.source_item_embedding = torch.nn.Embedding(self.total_num_items, self.embedding_size)

        self.target_user_embedding = torch.nn.Embedding(self.total_num_users, self.embedding_size)
        self.target_item_embedding = torch.nn.Embedding(self.total_num_items, self.embedding_size)

        with torch.no_grad():
            self.source_user_embedding.weight[self.overlapped_num_users: self.target_num_users].fill_(0)
            self.source_item_embedding.weight[self.overlapped_num_items: self.target_num_items].fill_(0)

            self.target_user_embedding.weight[self.target_num_users:].fill_(0)
            self.target_item_embedding.weight[self.target_num_items:].fill_(0)

        self.map_loss = nn.MSELoss()
        self.rec_loss = nn.TripletMarginLoss(margin=self.margin)

        # parameters initialization
        # self.apply(xavier_normal_initialization)

        if self.overlapped_num_users > 1:
            self.mode = 'overlap_users'
        elif self.overlapped_num_items > 1:
            self.mode = 'overlap_items'
        else:
            self.mode = 'non_overlap'

        if self.mode == 'overlap_users':
            self.user_interacted_items = self.build_interacted_items(dataset, mode='user')
        elif self.mode == 'overlap_items':
            self.item_interacted_users = self.build_interacted_items(dataset, mode='item')

    def build_interacted_items(self, dataset, mode='user'):
        
        if mode == 'user':
            interacted_items = [[] for _ in range(self.total_num_users)]
            for uid, iid in zip(dataset.source_user_ids.to_numpy(),
                                dataset.source_item_ids.to_numpy()):
                interacted_items[uid].append(iid)
            return interacted_items
        else:
            interacted_users = [[] for _ in range(self.total_num_items)]
            for iid, uid in zip(dataset.source_item_ids.to_numpy(),
                                dataset.source_user_ids.to_numpy()):
                interacted_users[iid].append(uid)
            return interacted_users
        

  

    def sample(self, ids, mode='user'):
        ids = ids.cpu().numpy()
        interacted = np.zeros_like(ids)
        non_interacted = np.zeros_like(ids)
        if mode =='user':
            all_candidates = list(range(self.total_num_items)) + \
                             list(range(self.target_num_items, self.total_num_items))
            for index, id in enumerate(ids):
                interacted_items = self.user_interacted_items[id]
                if len(interacted_items) == 0:
                    interacted_items.append(0)
                non_interacted_id = np.random.choice(all_candidates, size=1)[0]
                while non_interacted_id in interacted_items:
                    non_interacted_id = np.random.choice(all_candidates, size=1)[0]
                interacted[index] = np.random.choice(interacted_items, size=1)[0]
                non_interacted[index] = non_interacted_id
        else:
            all_candidates = list(range(self.overlapped_num_users)) + \
                             list(range(self.target_num_users, self.total_num_users))
            for index, id in enumerate(ids):
                interacted_users = self.item_interacted_users[id]
                if len(interacted_users) == 0:
                    interacted_users.append(0)
                non_interacted_id = np.random.choice(all_candidates, size=1)[0]
                while non_interacted_id in interacted_users:
                    non_interacted_id = np.random.choice(all_candidates, size=1)[0]
                interacted[index] = np.random.choice(interacted_users, size=1)[0]
                non_interacted[index] = non_interacted_id
        return torch.from_numpy(interacted).to(self.device), torch.from_numpy(non_interacted).to(self.device)

    @staticmethod
    def embedding_normalize(embeddings):
        emb_length = torch.sum(embeddings**2, dim=1, keepdim=True)
        ones = torch.ones_like(emb_length)
        norm = torch.where(emb_length > 1, emb_length, ones)
        return embeddings / norm

    @staticmethod
    def embedding_distance(emb1, emb2):
        return torch.sum((emb1-emb2)**2, dim=1)

    def set_phase(self, phase):
        self.phase = phase

    def calculate_source_loss(self,user_id, item_id, negative_item_id):

        source_user = user_id
        source_pos_item = item_id
        source_neg_item = negative_item_id

        source_user_e = self.source_user_embedding(source_user)
        source_pos_item_e = self.source_item_embedding(source_pos_item)
        source_neg_item_e = self.source_item_embedding(source_neg_item)

        loss_t = self.rec_loss(self.embedding_normalize(source_user_e),
                               self.embedding_normalize(source_pos_item_e),
                               self.embedding_normalize(source_neg_item_e))
        return loss_t

    def calculate_target_loss(self,user_id, item_id, negative_item_id):
        target_user = user_id
        target_pos_item = item_id
        target_neg_item = negative_item_id

        target_user_e = self.target_user_embedding(target_user)
        target_pos_item_e = self.target_item_embedding(target_pos_item)
        target_neg_item_e = self.target_item_embedding(target_neg_item)

        loss_t = self.rec_loss(self.embedding_normalize(target_user_e),
                               self.embedding_normalize(target_pos_item_e),
                               self.embedding_normalize(target_neg_item_e))
        return loss_t

    def calculate_map_loss(self, overlapping_user_ids):
        idx = overlapping_user_ids
        if self.mode == 'overlap_users':
            source_user_e = self.source_user_embedding(idx)
            target_user_e = self.target_user_embedding(idx)
            map_e = self.mapping_layer(source_user_e)
            loss_s = self.map_loss(map_e, target_user_e)
            source_pos_item, source_neg_item = self.sample(idx, mode='user')

            map_pos_item_e = self.mapping_layer(self.source_item_embedding(source_pos_item))
            map_neg_item_e = self.mapping_layer(self.source_item_embedding(source_neg_item))
            loss_u = self.rec_loss(self.embedding_normalize(target_user_e),
                                    self.embedding_normalize(map_pos_item_e),
                                    self.embedding_normalize(map_neg_item_e))
        else:
            source_item_e = self.source_item_embedding(idx)
            target_item_e = self.target_item_embedding(idx)
            map_e = self.mapping_layer(source_item_e)
            loss_s = self.map_loss(map_e, target_item_e)
            source_pos_user, source_neg_user = self.sample(idx, mode='item')

            map_pos_user_e = self.mapping_layer(self.source_user_embedding(source_pos_user))
            map_neg_user_e = self.mapping_layer(self.source_user_embedding(source_neg_user))
            loss_u = self.rec_loss(self.embedding_normalize(target_item_e),
                                   self.embedding_normalize(map_pos_user_e),
                                   self.embedding_normalize(map_neg_user_e))
        return loss_s + self.lamda * loss_u

    def predict(self,user_id,item_id):
        if self.phase == 'SOURCE':
            user = user_id
            item = item_id
            user_e = self.embedding_normalize(self.source_user_embedding(user))
            item_e = self.embedding_normalize(self.source_item_embedding(item))
            score = -self.embedding_distance(user_e, item_e)
        elif self.phase == 'TARGET':
            user = user_id
            item = item_id
            user_e = self.embedding_normalize(self.target_user_embedding(user))
            item_e = self.embedding_normalize(self.target_item_embedding(item))
            score = -self.embedding_distance(user_e, item_e)
        else:
            user = user_id
            item = item_id
            if self.mode == 'overlap_users':
                repeat_user = user.repeat(self.embedding_size, 1).transpose(0, 1)
                user_e = torch.where(repeat_user < self.overlapped_num_users, self.mapping_layer(self.source_user_embedding(user)),
                                     self.target_user_embedding(user))
                item_e = self.target_item_embedding(item)
            else:
                user_e = self.target_user_embedding(user)
                repeat_item = item.repeat(self.embedding_size, 1).transpose(0, 1)
                item_e = torch.where(repeat_item < self.overlapped_num_items, self.mapping_layer(self.source_item_embedding(item)),
                                     self.target_item_embedding(item))
        user_e = self.embedding_normalize(user_e)
        item_e = self.embedding_normalize(item_e)
        score = -self.embedding_distance(user_e, item_e)
        return score

    def full_sort_predict(self, user_id):
        if self.phase == 'SOURCE':
            user = user_id
            user_e = self.embedding_normalize(self.source_user_embedding(user))
            overlap_item_e = self.embedding_normalize(self.source_item_embedding.weight[:self.overlapped_num_items])
            source_item_e = self.embedding_normalize(self.source_item_embedding.weight[self.target_num_items:])
            all_item_e = torch.cat([overlap_item_e, source_item_e], dim=0)
        elif self.phase == 'TARGET':
            user = user_id
            user_e = self.embedding_normalize(self.target_user_embedding(user))
            all_item_e = self.embedding_normalize(self.target_item_embedding.weight[:self.target_num_items])
        else:
            user = user_id
            if self.mode == 'overlap_users':
                repeat_user = user.repeat(self.embedding_size, 1).transpose(0, 1)
                user_e = torch.where(repeat_user < self.overlapped_num_users, self.mapping_layer(self.source_user_embedding(user)),
                                     self.target_user_embedding(user))
                all_item_e = self.target_item_embedding.weight[:self.target_num_items]
            else:
                user_e = self.target_user_embedding(user)
                overlap_item_e = self.mapping_layer(self.source_item_embedding.weight[:self.overlapped_num_items])
                target_item_e = self.target_item_embedding.weight[self.overlapped_num_items:self.target_num_items]
                all_item_e = torch.cat([overlap_item_e, target_item_e], dim=0)
            user_e = self.embedding_normalize(user_e)
            all_item_e = self.embedding_normalize(all_item_e)

        num_batch_user, emb_dim = user_e.size()
        num_all_item, _ = all_item_e.size()
        dist = -2 * torch.matmul(user_e, all_item_e.permute(1, 0))
        dist += torch.sum(user_e ** 2, -1).view(num_batch_user, 1)
        dist += torch.sum(all_item_e ** 2, -1).view(1, num_all_item)
        return -dist.view(-1)

In [25]:
config = {"device":torch.device("cuda" if torch.cuda.is_available() else "cpu"),
          "embedding_size":64,
          "lambda":0.25,
          "margin":1,
          "mlp_hidden_size":[128,64]
        }

In [26]:
SSCDR = SSCDR(config,dataset)

In [27]:
epochs = 100

In [28]:
optimizer = torch.optim.Adam(SSCDR.parameters(), lr=0.001, weight_decay=0.0001)
source_loss =[]
epochs = 100
for epoch in range(epochs):
        SSCDR.train()
        train_loss = 0.0

        for i, data in enumerate(source_train_loader):
            user_id, item_id, negative_item_id, rating = data

            optimizer.zero_grad()
            
            # Calculating loss
            loss = SSCDR.calculate_source_loss(user_id, item_id, negative_item_id)
            print(f"Epoch: {epoch + 1}, Step: {i + 1}, Loss: {loss.item()}")

            train_loss += loss.item()

            loss.backward()
            optimizer.step()

        avg_train_loss = train_loss / len(source_train_loader)
        print(f"Average Training Loss for Epoch {epoch + 1}: {avg_train_loss}")
        source_loss.append(avg_train_loss)

Epoch: 1, Step: 1, Loss: 1.001672625541687
Epoch: 1, Step: 2, Loss: 0.9986277222633362
Epoch: 1, Step: 3, Loss: 1.003880500793457
Epoch: 1, Step: 4, Loss: 0.9998599290847778
Epoch: 1, Step: 5, Loss: 1.002308964729309
Epoch: 1, Step: 6, Loss: 0.9978358745574951
Epoch: 1, Step: 7, Loss: 0.9998515248298645
Epoch: 1, Step: 8, Loss: 1.0008790493011475
Epoch: 1, Step: 9, Loss: 1.0014501810073853
Epoch: 1, Step: 10, Loss: 1.0030205249786377
Epoch: 1, Step: 11, Loss: 0.9996251463890076
Epoch: 1, Step: 12, Loss: 0.9995733499526978
Epoch: 1, Step: 13, Loss: 1.0037890672683716
Epoch: 1, Step: 14, Loss: 0.9982880353927612
Epoch: 1, Step: 15, Loss: 1.0013569593429565
Epoch: 1, Step: 16, Loss: 0.9976283311843872
Epoch: 1, Step: 17, Loss: 1.0045801401138306
Epoch: 1, Step: 18, Loss: 0.9989111423492432
Epoch: 1, Step: 19, Loss: 1.0025289058685303
Average Training Loss for Epoch 1: 1.0008246302604675
Epoch: 2, Step: 1, Loss: 1.0039911270141602
Epoch: 2, Step: 2, Loss: 0.99740070104599
Epoch: 2, Step: 3

In [29]:
target_loss =[]

for epoch in range(epochs):
        SSCDR.train()
        train_loss = 0.0

        for i, data in enumerate(target_train_loader):
            user_id, item_id, negative_item_id, rating = data

            optimizer.zero_grad()
            
            # Calculating loss
            loss = SSCDR.calculate_target_loss(user_id, item_id, negative_item_id)
            print(f"Epoch: {epoch + 1}, Step: {i + 1}, Loss: {loss.item()}")

            train_loss += loss.item()

            loss.backward()
            optimizer.step()

        avg_train_loss = train_loss / len(source_train_loader)
        print(f"Average Training Loss for Epoch {epoch + 1}: {avg_train_loss}")
        target_loss.append(avg_train_loss)

Epoch: 1, Step: 1, Loss: 1.0043925046920776
Epoch: 1, Step: 2, Loss: 1.0074859857559204
Epoch: 1, Step: 3, Loss: 1.0034123659133911
Epoch: 1, Step: 4, Loss: 1.0021946430206299
Epoch: 1, Step: 5, Loss: 1.004373550415039
Epoch: 1, Step: 6, Loss: 1.0035725831985474
Epoch: 1, Step: 7, Loss: 1.0005640983581543
Average Training Loss for Epoch 1: 0.3697892490186189
Epoch: 2, Step: 1, Loss: 0.9945935010910034
Epoch: 2, Step: 2, Loss: 0.9938424825668335
Epoch: 2, Step: 3, Loss: 0.9951345920562744
Epoch: 2, Step: 4, Loss: 0.9858770370483398
Epoch: 2, Step: 5, Loss: 0.9848812818527222
Epoch: 2, Step: 6, Loss: 0.9822474718093872
Epoch: 2, Step: 7, Loss: 0.9808365702629089
Average Training Loss for Epoch 2: 0.3640743650888142
Epoch: 3, Step: 1, Loss: 0.9521310925483704
Epoch: 3, Step: 2, Loss: 0.9500268697738647
Epoch: 3, Step: 3, Loss: 0.9456632733345032
Epoch: 3, Step: 4, Loss: 0.9417223334312439
Epoch: 3, Step: 5, Loss: 0.9388534426689148
Epoch: 3, Step: 6, Loss: 0.9350574016571045
Epoch: 3, Ste

In [30]:
overlap_loss =[]


for epoch in range(epochs):
    SSCDR.train()
    train_loss = 0.0

  
    optimizer.zero_grad()

    # Calculate loss 
    loss = SSCDR.calculate_map_loss(overlap_tensor)
    print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")

    train_loss += loss.item()
    loss.backward()
    optimizer.step()

    avg_train_loss = train_loss / len(overlap_tensor)
    print(f"Average Training Loss for Epoch {epoch + 1}: {avg_train_loss}")
    overlap_loss.append(avg_train_loss)

Epoch: 1, Loss: 0.25594285130500793
Average Training Loss for Epoch 1: 0.00012797142565250396
Epoch: 2, Loss: 0.24547825753688812
Average Training Loss for Epoch 2: 0.00012273912876844406
Epoch: 3, Loss: 0.2353833019733429
Average Training Loss for Epoch 3: 0.00011769165098667145
Epoch: 4, Loss: 0.22513233125209808
Average Training Loss for Epoch 4: 0.00011256616562604904
Epoch: 5, Loss: 0.211689293384552
Average Training Loss for Epoch 5: 0.000105844646692276
Epoch: 6, Loss: 0.19831503927707672
Average Training Loss for Epoch 6: 9.915751963853836e-05
Epoch: 7, Loss: 0.18965259194374084
Average Training Loss for Epoch 7: 9.482629597187043e-05
Epoch: 8, Loss: 0.1799144148826599
Average Training Loss for Epoch 8: 8.995720744132996e-05
Epoch: 9, Loss: 0.17513920366764069
Average Training Loss for Epoch 9: 8.756960183382034e-05
Epoch: 10, Loss: 0.17359110713005066
Average Training Loss for Epoch 10: 8.679555356502533e-05
Epoch: 11, Loss: 0.1727425754070282
Average Training Loss for Epoch 1